# SigmaDock Runner — Four Usage Modes

This notebook demonstrates all four ways to run the SigmaDock runner:

| # | Mode | Input | Key params |
|---|------|-------|------------|
| 1 | Redocking from complex | PDB/CIF complex | `ligand_name` |
| 2 | Crossdocking from complex | PDB/CIF complex | `ligand_name`, `query_ligands` |
| 3 | Redocking from pre-extracted files | protein PDB + ligand SDF | `ligand_col` |
| 4 | Crossdocking from pre-extracted files | protein PDB + query SDFs + ref SDF | `ligand_col`, `ref_ligand_col` |

- Use `complex` for poses derived from co-predictions like **RFdiffusion** and **Boltz**, including ligands
- Use `pre-extracted` files only if you already have a protein and ligand that fit the pre-extracted format
- In most cases, just pass the complex structure to the SigmaDock runner

In [ ]:
from protflow.poses import Poses
from protflow.jobstarters import SbatchArrayJobstarter
from protflow.tools.sigmadock import SigmaDock
import os

# initialize the SigmaDock runner with an SbatchArrayJobstarter for GPU jobs
gpu_jobstarter = SbatchArrayJobstarter(max_cores=10, gpus=1, options="--time=00:10:00 ")
runner = SigmaDock(jobstarter=gpu_jobstarter)

# paths to all example files
COMPLEX_CIF   = "data/input_pdbs/sigmadock/1p0p_coc_complex.cif" # boltz output co prediction example
PROTEIN_PDB  = "data/input_pdbs/sigmadock/1p0p_ex_ligand.pdb" # structure without ligand
REF_LIGAND    = "data/input_pdbs/sigmadock/1p0p_coc_ligand.sdf" # ligand to structure above, correct coordinate placement
QUERY_LIGAND  = "data/input_pdbs/sigmadock/pet_ligand.sdf" # query ligand example
LIGAND_NAME   = "COC" # example ligand name
WORK_DIR      = "data/example_output/runners_example/sigmadock_outputs" 

os.makedirs(WORK_DIR, exist_ok=True)

---
## Mode 1 — Redocking from complex
The runner internally extracts the receptor PDB and reference ligand SDF from the complex, then re-docks the ligand back into the same receptor pocket.

This is the standard workflow and will be used most of the time for running SigmaDock in the ProtFlow pipeline.

**Use this** when providing poses derived from Boltz, AF3, or any other ligand–structure co-prediction tool.

In [ ]:
poses = Poses(poses=[COMPLEX_CIF], work_dir=WORK_DIR)

poses = runner.run(
    poses=poses,
    prefix="redock_from_complex",
    ligand_name=LIGAND_NAME,
    overwrite=True,
)

for col in poses.df.columns:
    if col.startswith("redock_from_complex"):
        print(f"{col}: {poses.df[col][0]}")

---
## Mode 2 — Crossdocking from complex

The runner still extracts the receptor and reference ligand from the complex internally.  
`query_ligands` provides the ligand(s) to dock, these are docked into the pocket defined by the reference ligand.

In [ ]:
poses = Poses(poses=[COMPLEX_CIF], work_dir=WORK_DIR)

poses = runner.run(
    poses=poses,
    prefix="crossdock_from_complex",
    ligand_name=LIGAND_NAME,
    query_ligands=[QUERY_LIGAND],   # absolute path recommended
    overwrite=True,
)

for col in poses.df.columns:
    if col.startswith("crossdock_from_complex"):
        print(f"{col}: {poses.df[col][0]}")

---
## Mode 3 — Redocking from pre-extracted files

Use this when you already have the receptor PDB and ligand SDF on disk.  
Initialize `Poses` with the receptor as the main pose, add the ligand as a column, and pass `ligand_col`.  
`split_complex` is skipped entirely.

In [ ]:
poses = Poses(poses=[PROTEIN_PDB], work_dir=WORK_DIR)
# specifically add the reference ligand as a column (e.g. from pre-extraction) - this will be used for pocket definition in redocking and as anchor in crossdocking;
poses.df["ligand_sdf"] = [REF_LIGAND] # needs to be passed as list

poses = runner.run(
    poses=poses,
    prefix="redock_from_extracted",
    ligand_col="ligand_sdf",
    overwrite=True,
)

for col in poses.df.columns:
    if col.startswith("redock_from_extracted"):
        print(f"{col}: {poses.df[col][0]}")

---
## Mode 4 — Crossdocking from pre-extracted files

Use this when structure, reference ligand, and query ligands are all pre extracted.  
`ligand_col` holds a **list** of query SDF paths per pose (crossdocking).  
`ref_ligand_col` holds the reference SDF used to define the binding pocket.

In [ ]:
poses = Poses(poses=[PROTEIN_PDB], work_dir=WORK_DIR)
poses.df["query_ligands"] = [[QUERY_LIGAND]]   # list per pose for crossdocking
poses.df["ref_ligand"] = [REF_LIGAND]

poses = runner.run(
    poses=poses,
    prefix="crossdock_from_extracted",
    ligand_col="query_ligands",
    ref_ligand_col="ref_ligand",
    overwrite=True,
)

for col in poses.df.columns:
    if col.startswith("crossdock_from_extracted"):
        print(f"{col}: {poses.df[col][0]}")